# SEG Regularization Tuning — ESOL

Minimal SEG benchmark to find optimal regularization settings.

In [1]:
# === Setup ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch

def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Workspace: {workspace_root}")

Workspace: c:\Users\robsc\Home\Dev\molfusion2


In [2]:
# === Configuration ===
# TUNE THESE for regularization experiments

CONFIG = {
    # === REPRODUCIBILITY ===
    "seed": 42,                   # Global seed for reproducibility (None to disable)
    
    # Architecture
    "hidden_channels": 64,
    "K": 3,
    "num_layers": 2,
    "pool": "sum",
    "set2set_processing_steps": 6,  # Number of LSTM steps for set2set pooling
    
    # Fusion
    "fusion": "cross_mha",           # Options: "concat", "cross_mha", "gated", "film"
    "fusion_dim": 64,
    "text_projection_dim": 64,  # Project 3072 → 64
    
    # === TEXT PROJECTION INITIALIZATION ===
    "text_proj_init": "xavier",      # "xavier" (scaled, recommended) or "kaiming" (PyTorch default)
    "text_proj_init_gain": 0.1,      # Gain for Xavier init (lower = smaller gradients)
    "freeze_text_proj": True,     
        # Freeze text projection (implicit regularization for small datasets)
    
    # === REGULARIZATION (tune these) ===
    "dropout": 0.3,              # Graph encoder dropout
    "fusion_dropout": 0.3,       # Fusion layer dropout
    "head_dropout": 0.3,         # Prediction head dropout
    "weight_decay": 1e-2,        # L2 regularization
    
    # === HEAD ===
    "head_type": "mlp",          # Options: "mlp" (default), "linear" (strong regularization)
    "head_hidden_dim": 32,      # Only used if head_type="mlp"
    
    # Training
    "learning_rate": 5e-4,
    "batch_size": 32,
    "num_epochs": 200,
    "patience": 30,
    
    # === LR SCHEDULER ===
    "scheduler": "plateau",      # Options: "plateau", "cosine", None
    "scheduler_patience": 5,     # Epochs without improvement before LR reduction
    "scheduler_factor": 0.5,     # Factor to reduce LR by
    "min_lr": 1e-6,              # Minimum learning rate
    
    # === GRADIENT CLIPPING ===
    "grad_clip": None,            # Max gradient norm (None to disable). Recommended: 1.0-5.0
}

# Set global seed for reproducibility
if CONFIG["seed"] is not None:
    set_seed(CONFIG["seed"])
    print(f"=== Seed: {CONFIG['seed']} (reproducibility enabled) ===")
else:
    print("=== Seed: None (non-deterministic) ===")

print("=== Regularization Settings ===")
print(f"Dropout: encoder={CONFIG['dropout']}, fusion={CONFIG['fusion_dropout']}, head={CONFIG['head_dropout']}")
print(f"Weight decay: {CONFIG['weight_decay']}")
print(f"Head: {CONFIG['head_type']}" + (f" (hidden={CONFIG['head_hidden_dim']})" if CONFIG['head_type'] == 'mlp' else " (no hidden layer)"))
print(f"LR: {CONFIG['learning_rate']}, batch: {CONFIG['batch_size']}")
print(f"Scheduler: {CONFIG['scheduler']} (patience={CONFIG['scheduler_patience']}, factor={CONFIG['scheduler_factor']})")
print(f"Pool: {CONFIG['pool']}" + (f" (steps={CONFIG['set2set_processing_steps']})" if CONFIG['pool'] == 'set2set' else ""))
print(f"Gradient clipping: {CONFIG['grad_clip']}")
print(f"Text proj init: {CONFIG['text_proj_init']} (gain={CONFIG['text_proj_init_gain']})")
print(f"Freeze text proj: {CONFIG['freeze_text_proj']}")

=== Seed: 42 (reproducibility enabled) ===
=== Regularization Settings ===
Dropout: encoder=0.3, fusion=0.3, head=0.3
Weight decay: 0.01
Head: mlp (hidden=32)
LR: 0.0005, batch: 32
Scheduler: plateau (patience=5, factor=0.5)
Pool: sum
Gradient clipping: None
Text proj init: xavier (gain=0.1)
Freeze text proj: True


In [3]:
# === Load ESOL Dataset ===
import deepchem as dc

tasks, datasets, _ = dc.molnet.load_delaney(featurizer='ECFP', splitter='scaffold')
train_dc, valid_dc, test_dc = datasets

train_smiles = list(train_dc.ids)
train_y = train_dc.y.reshape(-1).astype(np.float32)
valid_smiles = list(valid_dc.ids)
valid_y = valid_dc.y.reshape(-1).astype(np.float32)
test_smiles = list(test_dc.ids)
test_y = test_dc.y.reshape(-1).astype(np.float32)

print(f"Train: {len(train_smiles)} | Valid: {len(valid_smiles)} | Test: {len(test_smiles)}")
print(f"Target: mean={train_y.mean():.2f}, std={train_y.std():.2f}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Train: 902 | Valid: 113 | Test: 113
Target: mean=0.00, std=1.00


In [4]:
# === Load Text Embeddings (from cache) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_TEXT_DIR = workspace_root / "cache" / "cot_texts"
COT_EMB_DIR  = workspace_root / "cache" / "cot_embeddings"

TASK = "solubility_fast"  # Cache prefix — matches {task}_text_embeddings_compact.npz

# Load compact npz cache (run the conversion cell below first if only .pkl exists)
npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = train_smiles + valid_smiles + test_smiles
all_emb = cache.get_batch(all_smiles)               # (N, 3072) numpy float32
all_emb_t = torch.from_numpy(all_emb)                # → torch tensor

n_train = len(train_smiles)
n_valid = len(valid_smiles)
train_text_emb = all_emb_t[:n_train]
valid_text_emb = all_emb_t[n_train:n_train + n_valid]
test_text_emb  = all_emb_t[n_train + n_valid:]

print(f"✓ Loaded from {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={train_text_emb.shape}, valid={valid_text_emb.shape}, test={test_text_emb.shape}")

Loading embeddings from solubility_fast_text_embeddings_compact.npz...
  Loaded 1128 entries, dim=3072
  Memory mode: mapped
✓ Loaded from solubility_fast_text_embeddings_compact.npz (1128 molecules)
  Embeddings: train=torch.Size([902, 3072]), valid=torch.Size([113, 3072]), test=torch.Size([113, 3072])


In [5]:
# === Initialize SEG ===
from models import SEGPredictor, SEGPredictorConfig

# SEG config (graph + text fusion)
seg_config = SEGPredictorConfig(
    hidden_channels=CONFIG["hidden_channels"],
    K=CONFIG["K"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    pool=CONFIG["pool"],
    set2set_processing_steps=CONFIG["set2set_processing_steps"],
    text_embedding_dim=3072,
    text_projection_dim=CONFIG["text_projection_dim"],
    text_proj_init=CONFIG["text_proj_init"],
    text_proj_init_gain=CONFIG["text_proj_init_gain"],
    freeze_text_proj=CONFIG["freeze_text_proj"],
    fusion=CONFIG["fusion"],
    fusion_dim=CONFIG["fusion_dim"],
    fusion_dropout=CONFIG["fusion_dropout"],
    head_type=CONFIG["head_type"],
    head_hidden_dim=CONFIG["head_hidden_dim"],
    head_dropout=CONFIG["head_dropout"],
)

seg = SEGPredictor(config=seg_config)
print(f"SEG initialized with {CONFIG['fusion']} fusion, head={CONFIG['head_type']}")
print(f"Text proj: init={CONFIG['text_proj_init']}, frozen={CONFIG['freeze_text_proj']}")

SEG initialized with cross_mha fusion, head=mlp
Text proj: init=xavier, frozen=True


In [6]:
# === Train SEG ===
print(f"Training SEG (dropout={CONFIG['dropout']}, wd={CONFIG['weight_decay']}, seed={CONFIG['seed']})\n")

history = seg.fit(
    smiles_list=train_smiles,
    labels=train_y.tolist(),
    val_smiles=valid_smiles,
    val_labels=valid_y.tolist(),
    text_embeddings=train_text_emb,
    val_text_embeddings=valid_text_emb,
    num_epochs=CONFIG["num_epochs"],
    batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    patience=CONFIG["patience"],
    scheduler=CONFIG["scheduler"],
    scheduler_patience=CONFIG["scheduler_patience"],
    scheduler_factor=CONFIG["scheduler_factor"],
    min_lr=CONFIG["min_lr"],
    grad_clip=CONFIG["grad_clip"],
    seed=CONFIG["seed"] or 0,
    verbose=True,
)

# Count SEG parameters
n_params_seg = sum(p.numel() for p in seg._encoder.parameters())
n_params_seg += sum(p.numel() for p in seg._fusion.parameters())
n_params_seg += sum(p.numel() for p in seg._head.parameters())
if seg._text_proj: n_params_seg += sum(p.numel() for p in seg._text_proj.parameters())
print(f"\nSEG: {CONFIG['fusion']} fusion, {n_params_seg:,} parameters")

Training SEG (dropout=0.3, wd=0.01, seed=42)

Pre-computing molecular graphs for 902 molecules...
Pre-computed 902/902 valid graphs
Pre-computing molecular graphs for 113 molecules...
Pre-computed 113/113 valid graphs
Training SEGPredictor on 902 molecules...
Task: regression
Validation set: 113 molecules
Fusion method: cross_mha
LR scheduler: plateau
Epoch 001 | Train Loss: 0.9469 | Val RMSE: 0.8777 | LR: 5.00e-04
Epoch 005 | Train Loss: 0.2974 | Val RMSE: 0.6038 | LR: 5.00e-04
Epoch 010 | Train Loss: 0.2129 | Val RMSE: 0.5128 | LR: 5.00e-04
Epoch 015 | Train Loss: 0.1908 | Val RMSE: 0.5008 | LR: 5.00e-04
Epoch 020 | Train Loss: 0.1662 | Val RMSE: 0.4521 | LR: 5.00e-04
Epoch 025 | Train Loss: 0.1478 | Val RMSE: 0.4570 | LR: 5.00e-04
Epoch 030 | Train Loss: 0.1413 | Val RMSE: 0.4124 | LR: 5.00e-04
Epoch 035 | Train Loss: 0.1178 | Val RMSE: 0.4179 | LR: 2.50e-04
Epoch 040 | Train Loss: 0.1113 | Val RMSE: 0.4224 | LR: 2.50e-04
Epoch 045 | Train Loss: 0.1057 | Val RMSE: 0.4072 | LR: 1.25e

In [7]:
# === Test Set Evaluation ===
def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).reshape(-1), np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    ss_res, ss_tot = np.sum((y_true - y_pred) ** 2), np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

# SEG predictions
seg_preds = seg.predict_batch(test_smiles, text_embeddings=test_text_emb)
seg_metrics = regression_metrics(test_y, seg_preds)

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print()
print(f"{'Model':<20} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
print("-" * 52)
print(f"{'SEG (graph+text)':<20} {seg_metrics['rmse']:>10.4f} {seg_metrics['mae']:>10.4f} {seg_metrics['r2']:>10.4f}")
print("-" * 52)
print()
print(f"Config: dropout={CONFIG['dropout']}, wd={CONFIG['weight_decay']}, fusion={CONFIG['fusion']}, pool={CONFIG['pool']}")

TEST SET RESULTS

Model                      RMSE        MAE         R²
----------------------------------------------------
SEG (graph+text)         0.3922     0.3126     0.8538
----------------------------------------------------

Config: dropout=0.3, wd=0.01, fusion=cross_mha, pool=sum
